In [3]:
import pypsa
import pandas as pd

# --- 1. ΑΡΧΙΚΟΠΟΙΗΣΗ ΔΙΚΤΥΟΥ & ΧΡΟΝΟΥ ---
network = pypsa.Network()

# Ορίζουμε 8 χρονικά βήματα (π.χ. για 8 ώρες) για να ταιριάζουν τα δεδομένα μας
network.set_snapshots(range(8)) 

# --- 1β. ΔΗΛΩΣΗ ΤΩΝ CARRIERS (ΛΥΝΕΙ ΤΑ WARNINGS) ---
network.add("Carrier", "AC")     # Εναλλασσόμενο ρεύμα
network.add("Carrier", "gas", co2_emissions=0.2)  # Φυσικό αέριο (π.χ. με 0.2 τόνους CO2/MWh)
network.add("Carrier", "solar", co2_emissions=0.0) # Ήλιος (0 εκπομπές)

# --- 2. ΔΗΜΙΟΥΡΓΙΑ ΖΥΓΟΥ (BUS) ---
# Φτιάχνουμε τον κεντρικό κόμβο όπου θα συνδεθούν όλα
network.add("Bus", "Transmission_Node_Athens", v_nom=380, carrier="AC")

# --- 3. ΔΗΜΙΟΥΡΓΙΑ ΠΑΡΑΓΩΓΗΣ (GENERATORS) ---

# 3α. Συμβατική Μονάδα (Φυσικό Αέριο) - Σταθερή αλλά με κόστος καυσίμου
network.add(
    "Generator",
    "Gas_Plant_1",
    bus="Transmission_Node_Athens",
    carrier="gas",
    p_nom=500,               # 500 MW μέγιστη ισχύς
    marginal_cost=80,        # 80 €/MWh
    capital_cost=100000,
    p_nom_extendable=False   # Σταθερή χωρητικότητα
)

# 3β. Ανανεώσιμη Πηγή (Φωτοβολταϊκό) - Μεταβαλλόμενη αλλά δωρεάν
solar_profile = pd.Series([0.0, 0.0, 0.2, 0.8, 1.0, 0.7, 0.1, 0.0]) 

network.add(
    "Generator",
    "Solar_Park_1",
    bus="Transmission_Node_Athens",
    carrier="solar",
    p_nom=200,               # 200 MW εγκατεστημένη ισχύς
    p_max_pu=solar_profile,  # Η διαθεσιμότητα βάσει ήλιου (0% έως 100%)
    marginal_cost=0,         # Μηδενικό οριακό κόστος
    p_nom_extendable=True    # Επιτρέπουμε στο μοντέλο να "χτίσει" κι άλλο αν χρειαστεί
)

# --- 4. ΔΗΜΙΟΥΡΓΙΑ ΚΑΤΑΝΑΛΩΣΗΣ (LOAD) ---
# Η ζήτηση (σε MW) για τις ίδιες 8 ώρες
demand_profile = pd.Series([250, 300, 450, 500, 480, 550, 400, 280])

network.add(
    "Load",
    "Athens_Demand",
    bus="Transmission_Node_Athens",
    p_set=demand_profile
)

# --- 5. ΕΠΙΛΥΣΗ (OPTIMIZATION) ---
# Ζητάμε από το PyPSA να καλύψει τη ζήτηση με το ελάχιστο δυνατό κόστος
print("Ξεκινάει η επίλυση του συστήματος...\n")
network.optimize()

# --- 6. ΕΜΦΑΝΙΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ ---
print("\n--- ΑΠΟΤΕΛΕΣΜΑΤΑ ΠΑΡΑΓΩΓΗΣ ΑΝΑ ΩΡΑ (σε MW) ---")
print(network.generators_t.p)

Ξεκινάει η επίλυση του συστήματος...



INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.07s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 17 primals, 41 duals
Objective: 6.64e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.



--- ΑΠΟΤΕΛΕΣΜΑΤΑ ΠΑΡΑΓΩΓΗΣ ΑΝΑ ΩΡΑ (σε MW) ---
name      Gas_Plant_1  Solar_Park_1
snapshot                           
0               250.0          -0.0
1               300.0          -0.0
2                -0.0         450.0
3                -0.0         500.0
4                -0.0         480.0
5                -0.0         550.0
6                -0.0         400.0
7               280.0          -0.0
